# Baseline de Recomendación

Antes de usar bandits, primero conviene tener referencias simples. El EDA ya nos sugirio que un solo canal global probablemente se quede corto porque la mejor opcion cambia por contexto.


## Que queremos contrastar

- baseline global: un solo canal para todo
- baseline contextual: mejor canal histórico por `audiencia + objetivo`
- luego comparar eso contra el bandit


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.data.carga_datos import cargar_dataset
from src.data.limpieza import limpiar_dataset_publicidad
from src.data.features import crear_contexto
from src.modelos.baseline import baseline_mejor_canal_global, baseline_mejor_canal_por_contexto
from src.modelos.filtro_colaborativo import analizar_viabilidad_filtrado_colaborativo


In [2]:
df = cargar_dataset(ROOT / 'social_media_ads_filtered.csv')
df = crear_contexto(limpiar_dataset_publicidad(df))

baseline_global = baseline_mejor_canal_global(df)
baseline_contextual = baseline_mejor_canal_por_contexto(df)
analisis_cf = analizar_viabilidad_filtrado_colaborativo()

print('Baseline global:')
print(baseline_global)

print('\nViabilidad de filtrado colaborativo:')
print(analisis_cf)


Baseline global:
{'Channel_Used': 'Facebook', 'roi_promedio': 3.9951102941176475, 'conversion_promedio': 0.08048713235294118}

Viabilidad de filtrado colaborativo:
{'aplica_directamente': False, 'motivo': 'El dataset no contiene interacciones usuario-item repetidas ni una matriz explícita de usuarios contra canales o anuncios.', 'adaptacion_sugerida': 'Usar agregación por contexto (audiencia + objetivo) y comparar canales como brazos del bandit.'}


In [3]:
baseline_contextual.head(20)


,Contexto,Channel_Used,roi_promedio,conversion_promedio,costo_promedio
0,All Ages | Brand Awareness,Twitter,4.288095,0.070476,7570.991429
4,All Ages | Increase Sales,Twitter,4.176957,0.081304,6877.010000
8,All Ages | Market Expansion,Facebook,4.598125,0.083750,7575.532187
12,All Ages | Product Launch,Instagram,4.495385,0.081923,6354.018462
16,Men 18-24 | Brand Awareness,Facebook,4.187500,0.083929,7079.548214
20,Men 18-24 | Increase Sales,Instagram,4.221034,0.084138,7205.264138
24,Men 18-24 | Market Expansion,Twitter,4.058000,0.083600,9489.632800
28,Men 18-24 | Product Launch,Instagram,4.585789,0.083158,5266.302632
32,Men 25-34 | Brand Awareness,Facebook,4.529000,0.073667,8222.873333
36,Men 25-34 | Increase Sales,Instagram,4.347586,0.076897,7116.706207


## Conclusión

Lo que ya se empieza a notar aca es que el baseline global sirve como referencia, pero no captura la variacion real entre segmentos. En cambio el baseline contextual ya conversa mucho mejor con lo que vimos en el EDA, porque reconoce que no todas las audiencias responden igual ni persiguen el mismo objetivo.
